# Visualzing points of interest on map

In [1]:
# !pip install folium
import folium

# Set up KFUPM as a source node
source_point = (26.3071, 50.1459)

# Set up King Abdulaziz Center for World Culture - Ithra as a destination node
destination_point = (26.3354, 50.121)

# Set up the map reference point as the midpoint between the selected points
reference_point = ((source_point[0]+destination_point[0])/2, (source_point[1]+destination_point[1])/2)

m = folium.Map(location=reference_point, zoom_start=13.6,scrollWheelZoom=True, dragging=True)
folium.Marker(location=source_point,icon=folium.Icon(color='blue',icon='graduation-cap', prefix='fa'), tooltip="KFUPM").add_to(m)
folium.Marker(location=destination_point,icon=folium.Icon(color='green',icon='book', prefix='fa'), tooltip="King Abdulaziz Center for World Culture - Ithra").add_to(m)

m

# Find the shortest path between two points of interest

Find and render the routes between the two selected points using BFS, DFS, Dijkstra, UCS, and bidirectional Dijkstra. Compare these routing algorithms in terms of time and route length. BFS minimizes the number of graph edges and DFS returns the first path it finds; neither is guaranteed to minimize distance. Dijkstra and UCS should minimize the edge-length cost.

In [2]:
#!pip install osmnx
#!pip install optalgotools
import osmnx
from optalgotools.structures import Node
from optalgotools.routing import cost, draw_route
from optalgotools.algorithms.graph_search import BFS, DFS, Dijkstra, UCS, Bidirectional_Dijkstra
import pandas as pd
pd.options.compute.use_numexpr = False # for pandas ≥ 1.0, Disable numexpr for all subsequent pandas eval/query operations

# Download a graph that actually contains both POIs. The old dist=300 graph
# covered only 300 m around the midpoint, although the POIs are kilometres apart.
buffer_degrees = 0.01  # approximately 1 km of padding around the two POIs
bbox = (
    min(source_point[1], destination_point[1]) - buffer_degrees,  # left/west
    min(source_point[0], destination_point[0]) - buffer_degrees,  # bottom/south
    max(source_point[1], destination_point[1]) + buffer_degrees,  # right/east
    max(source_point[0], destination_point[0]) + buffer_degrees,  # top/north
)
G = osmnx.graph_from_bbox(
    bbox,
    network_type="all",
    simplify=True,
)

# Get the OSM IDs of the nearest graph nodes (X=longitude, Y=latitude).
origin_id = osmnx.distance.nearest_nodes(
    G, X=source_point[1], Y=source_point[0]
)
destination_id = osmnx.distance.nearest_nodes(
    G, X=destination_point[1], Y=destination_point[0]
)

# Verify that each POI snapped to a genuinely nearby graph node.
origin_data = G.nodes[origin_id]
destination_data = G.nodes[destination_id]
origin_snap_m = osmnx.distance.great_circle(
    source_point[0], source_point[1], origin_data["y"], origin_data["x"]
)
destination_snap_m = osmnx.distance.great_circle(
    destination_point[0], destination_point[1],
    destination_data["y"], destination_data["x"]
)
print(f"Origin snap distance: {origin_snap_m:.1f} m")
print(f"Destination snap distance: {destination_snap_m:.1f} m")
straight_line_m = osmnx.distance.great_circle(
    source_point[0], source_point[1],
    destination_point[0], destination_point[1]
)
print(f"Straight-line POI distance: {straight_line_m:.1f} m")
if max(origin_snap_m, destination_snap_m) > 500:
    raise ValueError("A POI is more than 500 m from the downloaded network.")

# Convert the source and destination nodes to Node
origin = Node(graph=G, osmid=origin_id)
destination = Node(graph=G, osmid=destination_id)

Origin snap distance: 13.7 m
Destination snap distance: 185.8 m
Straight-line POI distance: 4007.7 m


## BFS

In [3]:
solution = BFS(origin, destination)
route = solution.result
print(f"Cost: {cost(G,route)} m")
print(f"Process time: {solution.time} s")
print(f"Space required: {solution.space} bytes")
print(f"Explored nodes: {solution.explored}")
draw_route(G,route)

Cost: 10046.8247 m
Process time: 0.171875 s
Space required: 2344 bytes
Explored nodes: 2162


## DFS

In [4]:
solution = DFS(origin, destination)
route = solution.result
print(f"Cost: {cost(G,route)} m")
print(f"Process time: {solution.time} s")
print(f"Space required: {solution.space} bytes")
print(f"Explored nodes: {solution.explored}")
draw_route(G,route)

Cost: 39575.1513 m
Process time: 0.140625 s
Space required: 1816 bytes
Explored nodes: 1517


## Dijkstra algorithm

In [5]:
unrelaxed_nodes = [Node(G, osmid) for osmid in G.nodes()]
solution = Dijkstra(origin, destination, unrelaxed_nodes)
route = solution.result
print(f"Cost: {cost(G,route)} m")
print(f"Process time: {solution.time} s")
print(f"Space required: {solution.space} bytes")
print(f"Explored nodes: {solution.explored}")
draw_route(G,route)

Cost: 7207.7582 m
Process time: 5.46875 s
Space required: 29336 bytes
Explored nodes: 3157


## UCS

In [6]:
from optalgotools.algorithms.graph_search import BFS, DFS, Dijkstra, UCS, Bidirectional_Dijkstra

solution = UCS(origin, destination)
route = solution.result
print(f"Cost: {cost(G,route)} m")
print(f"Process time: {solution.time} s")
print(f"Space required: {solution.space} bytes")
print(f"Explored nodes: {solution.explored}")
draw_route(G,route)

Cost: 7207.7582 m
Process time: 2.53125 s
Space required: 1464 bytes
Explored nodes: 3157


## Bidirectional Dijkstra

In [7]:
unrelaxed_nodes = [Node(G,osmid) for osmid in G.nodes()]
solution = Bidirectional_Dijkstra(origin, destination, unrelaxed_nodes)
route = solution.result
print(f"Cost: {cost(G,route)} m")
print(f"Process time: {solution.time} s")
print(f"Space required: {solution.space} bytes")
print(f"Explored nodes: {solution.explored}")
draw_route(G,route)

Cost: 7207.7582 m
Process time: 4.109375 s
Space required: 29336 bytes
Explored nodes: 1520
